# Δ25-hTS / dUMP — a non-cooperative two-site ITC fit (public data)

Three dUMP-into-Δ25 (N-terminally truncated) human thymidylate synthase isotherms, fit jointly with a
cooperative (sequential two-site) model. The data are the **public** eLife 2022 source data
(Bonin, Sapienza & Lee, *Dynamic allostery in substrate binding by human thymidylate synthase*,
eLife 11:e79915; Appendix 1 — Figure 3; also Dryad doi:10.5061/dryad.j9kd51cfx).

Unlike full-length hTS/dUMP (~9-fold positive cooperativity; Bonin et al. 2019), the Δ25 truncation
abolishes cooperativity, so the fit returns `delta_delta_g ≈ 0` (cooperativity ratio ≈ 1, ΔG1 ≈ ΔG2).

Run the fit first, then this notebook only loads the saved samples:

```bash
python examples/bonin_2022_delta25_dump/fit.py --full
```

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Locate this example directory whether the notebook is launched from here or the repo root.
EXAMPLE_DIR = Path.cwd()
if not (EXAMPLE_DIR / "results").exists():
    candidate = EXAMPLE_DIR / "examples" / "bonin_2022_delta25_dump"
    if candidate.exists():
        EXAMPLE_DIR = candidate
sys.path.insert(0, str(EXAMPLE_DIR.parent))  # so `import _common` works

import _common
from bayesian_binding.regression import summarize_cooperative_posterior

samples, summary, metadata = _common.load_results(EXAMPLE_DIR / "results")
print("datasets:", metadata["datasets"])
print("mode:", metadata["mode"], "| chains x draws:", metadata["mcmc"])
summary

## Cooperative-model posterior summary

`delta_g` is the first microscopic binding free energy and `delta_g + delta_delta_g` the second; the cooperativity ratio is `exp(-delta_delta_g / RT)` (1 = no cooperativity).

In [ ]:
post = summarize_cooperative_posterior(samples)
ddg = np.asarray(samples["delta_delta_g"], dtype=float)

rows = []
for label, key in [
    ("ΔG1 (kcal/mol)", "delta_g1_kcal_per_mol"),
    ("ΔG2 (kcal/mol)", "delta_g2_kcal_per_mol"),
    ("ΔH1 (kcal/mol)", "delta_h1_kcal_per_mol"),
    ("ΔH2 (kcal/mol)", "delta_h2_kcal_per_mol"),
    ("cooperativity ratio", "cooperativity_ratio"),
]:
    pi = post[key]
    rows.append({"quantity": label, "mean": pi["mean"], "2.5%": pi["lower"], "97.5%": pi["upper"]})
rows.append({
    "quantity": "ΔΔG (kcal/mol)",
    "mean": float(ddg.mean()),
    "2.5%": float(np.quantile(ddg, 0.025)),
    "97.5%": float(np.quantile(ddg, 0.975)),
})
pd.DataFrame(rows).set_index("quantity").round(3)

## Conclusion: no cooperativity

In [ ]:
ratio = post["cooperativity_ratio"]["mean"]
dg1 = post["delta_g1_kcal_per_mol"]["mean"]
dg2 = post["delta_g2_kcal_per_mol"]["mean"]
print(f"cooperativity ratio = {ratio:.2f}   (1 = none; full-length hTS/dUMP is ~9)")
print(f"ΔΔG                 = {float(ddg.mean()):+.2f} kcal/mol   (0 = none)")
print(f"ΔG1 ≈ ΔG2           = {dg1:.2f} ≈ {dg2:.2f} kcal/mol")
print("=> The Δ25 N-terminal truncation abolishes the positive cooperativity of full-length hTS/dUMP.")